# Troubleshooting Log — GSE127465 scRNA-seq Pipeline

이 노트북은 파이프라인 구축 과정에서 발생한 문제들의 **발견 -> 원인 분석 -> 해결** 흐름을 기록한다.  
최종 파이프라인(`01_geo_pipeline.ipynb`)에 반영된 모든 설계 결정의 근거가 여기에 있다.

---

## Issues Index

| # | 문제 | 발견 시점 | 해결 방법 |
|---|------|----------|----------|
| 1 | Barcode 중복 | 샘플 병합 직후 | `index_unique='-'` |
| 2 | 메모리 에러 (RAM 32GB) | 샘플 로딩 중 | CSR sparse + 즉시 `del df` |
| 3 | MAD 필터가 아무것도 잡지 못함 | 첫 번째 QC 결과 | Hard cutoff 선적용 후 MAD |
| 4 | 클러스터 127개 생성 (과분할) | Leiden 클러스터링 | QC 재진행으로 노이즈 세포 제거 |
| 5 | Batch effect (클러스터 0에 편중) | UMAP 시각화 | Harmony batch correction |
| 6 | Harmony `ValueError` (shape mismatch) | Harmony 적용 중 | `harmonypy` 직접 호출로 우회 |
| 7 | HVG 필터 후 marker gene 조회 불가 | Annotation 단계 | `adata.raw` 백업 위치 수정 |


---
## Issue 1 — Barcode 중복 문제

### 현상
26개 샘플을 `sc.concat`으로 병합하자 `UserWarning: utils.warn_names_duplicates("obs")` 경고 발생.

### 원인
각 샘플이 독립적으로 sequencing되기 때문에 서로 다른 샘플에서 동일한 barcode 시퀀스가 등장할 수 있다.  
ex) sample A의 `bcIHWD`와 sample B의 `bcIHWD`가 충돌.

### 해결
`sc.concat`의 `index_unique` 파라미터로 샘플 번호를 barcode에 접미사로 붙여 고유화.

In [ ]:
# ❌ 문제 코드
adata_raw = sc.concat(adata_list, join='outer')
# -> UserWarning: utils.warn_names_duplicates("obs")

# ✅ 해결 코드
adata_raw = sc.concat(adata_list, join='outer', index_unique='-')
# bcIHWD -> bcIHWD-0 (sample 0), bcIHWD-1 (sample 1)
# 이후 sample 컬럼 추출 시: adata.obs_names.str.split('-').str[-1]
print("index_unique='-' 적용 후 barcode 예시:")
# print(adata_raw.obs_names[:5].tolist())

---
## Issue 2 — 메모리 에러 (RAM 32GB 환경)

### 현상
26개 파일 로딩 중 커널 크래시.  
`The Kernel crashed while executing code in the current cell.`

### 원인
26 samples × ~5,000 cells × 41,861 genes = **약 5.4TB dense float64 행렬**

### 해결 (3단계)
1. `float32` 캐스팅: float64 대비 메모리 50% 절감 (정밀도 손실은 count 데이터에서 무시 가능)
2. CSR sparse matrix: scRNA-seq는 대부분 값이 0 -> 비영 좌표+값만 저장
3. 루프마다 `del df`: df를 즉시 GC에 반환

In [ ]:
# ❌ 문제 코드 — Dense float64로 로딩
import pandas as pd
import scanpy as sc

adata_list = []
for f in files:
    df = pd.read_csv(f, sep='\t', index_col=0, compression='gzip')
    adata = sc.AnnData(df.T)          # Dense matrix, float64
    adata_list.append(adata)
# -> Kernel crash (OOM)

# ✅ 해결 코드
import scipy.sparse as sp

adata_list = []
for f in files:
    df = pd.read_csv(f, sep='\t', index_col=0, compression='gzip')
    adata = sc.AnnData(sp.csr_matrix(df.values.astype('float32')))  # sparse + float32 적용
    adata.obs_names = df.index.tolist()
    adata.var_names = df.columns.tolist()
    adata_list.append(adata)
    del df  # 메모리 즉시 해제
print("CSR sparse + float32 + del df 조합으로 메모리 절감")

### CSR vs COO vs CSC 비교

| 형식 | 구조 | 적합한 용도 |
|------|------|------------|
| COO (Coordinate) | (행, 열, 값) 좌표 단순 기록 | 데이터 초기 생성 |
| **CSR** (Compressed Sparse Row) | 행 중심 압축, 연산 속도 빠름 | **분석/학습 (권장)** |
| CSC (Compressed Sparse Column) | 열 중심 압축 | 특정 gene 슬라이싱 |

Scanpy의 내부 연산 대부분이 행(cell) 단위로 이루어지므로 CSR이 적합하다.

---
## Issue 3 — MAD 필터가 아무것도 잡지 못함 (The MAD Trap)

### 현상
첫 번째 QC 실행 결과(MAD만 적용):
```
outlier
False    173954
```
`total_counts`와 `n_genes_by_counts` 기준 이상치가 **단 하나도 없음**.

### 원인 분석
바이올린 플롯을 보면 `n_genes_by_counts`와 `total_counts`가 0 근처에 점이 밀집되어 있다.  
low-count 세포가 데이터의 다수를 차지 -> **median 자체가 낮게 형성** -> low-count 세포도 "median 근처"로 인식 -> outlier 아님 판정.

MAD는 데이터의 분포 기준으로 자동 임계값을 잡기 때문에, 데이터 자체가 노이즈로 오염되어 있으면 노이즈를 정상으로 판정한다.

### 해결
Hard cutoff를 MAD를 적용하기 **전에** 선적용하여 median을 끌어올린다.

In [ ]:
import scanpy as sc
import numpy as np
from scipy.stats import median_abs_deviation

# ❌ 문제 코드 — MAD 단독 적용
def is_outlier(adata, metric, nmads):
    M = adata.obs[metric]
    return (M < np.median(M) - nmads * median_abs_deviation(M)) |            (np.median(M) + nmads * median_abs_deviation(M) < M)

adata = sc.read_h5ad('dataset/raw/GSE127465_human_all.h5ad')
# ... (QC metrics 계산 생략)
# adata.obs['outlier'] = is_outlier(adata, 'log1p_total_counts', 5) | ...
# print(adata.obs['outlier'].value_counts())
# -> False 173954  <- low-count 세포가 전부 통과됨

# ✅ 해결 코드 — Hard cutoff 선적용 후 MAD
# Step 1: Hard cutoff -> median을 끌어올림
adata = adata[adata.obs['n_genes_by_counts'] >= 200].copy()
adata = adata[adata.obs['total_counts']      >= 500].copy()
print(f'하드 컷오프 후: {adata.n_obs}')  # 173954 -> 49921

# Step 2: 이제 MAD가 제대로 작동
# adata.obs['outlier'] = is_outlier(adata, 'log1p_total_counts', 5) | ...
# print(adata.obs['outlier'].value_counts())
# -> True 135  <- 정상적으로 극단값 포착

print("핵심: Hard cutoff로 median을 끌어올린 뒤 MAD를 적용해야 한다.")
print("순서: n_genes >= 200, total_counts >= 500 -> MAD (5 nmads)")

---
## Issue 4 — 클러스터 127개 생성 (과분할)

### 현상
첫 번째 QC 이후 Leiden clustering 결과:
```
클러스터 수: 127
클러스터 0: 108,574개 (전체의 99%)
나머지 126개 클러스터: 각 수십 개
```

### 원인 분석
Issue 3의 MAD trap으로 인해 노이즈 세포(low-count, 사멸 세포)가 대량으로 남아있는 상태로 클러스터링이 진행됨.  
노이즈 세포들이 서로 비슷하게 묶여 거대한 클러스터 0을 형성하고, 나머지 소수 세포들이 각각 독립 클러스터로 분리됨.

`resolution`을 아무리 낮춰도 해결되지 않음 -> resolution 문제가 아닌 **데이터 품질 문제**.

### 해결
Issue 3의 hard cutoff + MAD 필터를 올바르게 적용한 QC 재진행.

In [ ]:
# ❌ 문제 결과 재현 (실제 실행 X — 기록용)
# resolution=0.5  -> 클러스터 127개, cluster 0에 108,574개 편중
# resolution=0.1  -> 클러스터 90개,  cluster 0에 111,377개 편중
# resolution=0.005 -> 클러스터 129개, cluster 0에 108,574개 편중
# -> resolution을 어떻게 바꿔도 동일한 패턴 -> 데이터 문제임을 확인

# ✅ QC 재진행 후 결과
# resolution=0.5 -> 클러스터 15개, 고르게 분포
# -> 정상적인 클러스터링 확인

print("진단 기준:")
print("  정상: 여러 클러스터에 세포가 고르게 분포")
print("  비정상: 특정 클러스터 1개에 전체의 90%+ 편중")
print()
print("resolution 조정으로 해결되지 않으면 -> 데이터 품질 문제 의심")

---
## Issue 5 — Batch Effect (샘플 출처 기반 클러스터링)

### 현상
QC 재진행 후에도 클러스터가 생물학적 세포 타입이 아닌 **샘플 번호** 기준으로 형성됨.  
`sc.pl.umap(adata, color='sample')`으로 시각화 시 클러스터가 샘플별로 분리되어 있음.

### 원인
26개 샘플은 서로 다른 환자/수술 시점/실험 배치에서 수집됨.  
동일한 세포 타입이라도 샘플마다 기술적 노이즈(sequencing depth, library preparation 차이)가 달라  
생물학적 신호보다 기술적 차이가 클러스터링을 왜곡.

### 해결
Harmony로 PCA 임베딩을 샘플별로 반복 보정 -> sample-agnostic한 임베딩 생성

In [ ]:
# Batch effect 진단 코드
sc.pl.umap(adata, color='sample')
# -> 클러스터가 샘플 번호와 일치하면 batch effect 존재

# ✅ Harmony 적용
import harmonypy as hm

# 샘플 정보 추출
adata.obs['sample'] = adata.obs_names.str.split('-').str[-1]

# Harmony: PCA 임베딩을 샘플별로 보정
pca_result = adata.obsm['X_pca']
meta       = adata.obs[['sample']]
ho         = hm.run_harmony(pca_result, meta, 'sample')
adata.obsm['X_pca_harmony'] = ho.Z_corr

# Harmony 적용 결과 검증
sc.pp.neighbors(adata, use_rep='X_pca_harmony', n_pcs=17)
sc.tl.umap(adata, random_state=42)
sc.pl.umap(adata, color='sample')
# -> 샘플이 고르게 섞이면 batch correction 성공

print("Harmony 적용 전: 샘플별로 클러스터 분리")
print("Harmony 적용 후: 동일 세포 타입끼리 샘플 무관하게 군집")

---
## Issue 6 — Harmony `ValueError`: shape mismatch

### 현상
`scanpy.external`의 `sce.pp.harmony_integrate` 사용 시:
```
ValueError: Value passed for key 'X_pca_harmony' is of incorrect shape.
Values of obsm must match dimensions ('obs',) of parent.
Value had shape (50,) while it should have had (115909,).
```

### 원인 분석
클러스터링 문제 발견 후 `sample` 컬럼을 obs에 추가하면서,  
이전 HVG 필터링 기준으로 생성된 `adata.obsm['X_pca']`와 현재 adata의 shape가 불일치.  
`sce.pp.harmony_integrate`의 내부 shape 검증 로직이 이를 잡아냄.

추가로 `harmonypy` 버전에 따라 `sce.pp.harmony_integrate`가 shape를 잘못 처리하는 버그가 존재.

### 해결
`scanpy.external` 래퍼 대신 `harmonypy`를 직접 호출.

In [ ]:
# ❌ 문제 코드
# import scanpy.external as sce
# sce.pp.harmony_integrate(adata, 'sample')
# -> ValueError: shape mismatch

# ✅ 해결 코드 — harmonypy 직접 호출
import harmonypy as hm

# PCA 재실행 (shape 동기화)
sc.tl.pca(adata, svd_solver='arpack')
print(adata.obsm['X_pca'].shape)  # (n_cells, 50) 확인

# harmonypy 직접 호출
pca_result = adata.obsm['X_pca']         # (n_cells, 50)
meta = adata.obs[['sample']]             # (n_cells, 1)
ho = hm.run_harmony(pca_result, meta, 'sample')
adata.obsm['X_pca_harmony'] = ho.Z_corr  # (n_cells, 50)

print("sce.pp.harmony_integrate 대신 hm.run_harmony 직접 호출로 우회")
print("결과는 동일하나 shape 검증 로직을 통과함")

---
## Issue 7 — HVG 필터 후 Marker Gene 조회 불가

### 현상
Annotation 단계에서 marker gene 발현량 확인 시:
```python
sc.pl.umap(adata, color=['CD3D', 'CD68', 'EPCAM', 'CD19'])
```
```
KeyError: 'CD3D' is not in adata.var_names
```
`'CD3D' in adata.var_names -> False`

### 원인
HVG 필터 적용 후 `adata = adata[:, adata.var.highly_variable].copy()`로 2,000개 유전자만 남겼는데,  
`CD3D`가 HVG 2,000개에 포함되지 않았음.

`adata.raw` 백업을 HVG 필터 **이후**에 실행하는 실수를 했기 때문.

### 해결
`adata.raw = adata`를 HVG 필터 적용 **직전**에 실행.  
`sc.pl.umap`은 `use_raw=True`(기본값)로 `adata.raw`에서 유전자를 조회하므로 HVG 외 유전자도 확인 가능.

In [ ]:
# ❌ 문제 코드 — raw 백업 위치가 잘못됨
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3', layer='counts')
adata = adata[:, adata.var.highly_variable].copy()  # ← 이미 2000개로 잘림
adata.raw = adata  # ← 2000개짜리를 백업해서 의미 없음

# ✅ 해결 코드 — HVG 필터 직전에 백업
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3', layer='counts')
adata.raw = adata                                    # ← 41,861개 전체 백업
adata = adata[:, adata.var.highly_variable].copy()  # ← 이후 2,000개로 필터

# annotation 시 HVG 외 유전자 조회
sc.pl.umap(adata, color=['CD3D', 'CD68'], use_raw=True)  # adata.raw에서 조회

print("adata.raw 백업은 반드시 HVG 필터 적용 직전에 실행")
print("sc.pl.umap의 use_raw=True (기본값)가 adata.raw에서 유전자를 조회함")

---
## Summary: 최종 파이프라인에 반영된 설계 결정

| 설계 결정 | 근거 (이슈 번호) |
|----------|----------------|
| `index_unique='-'` 사용 | Issue 1: barcode 중복 방지 |
| CSR sparse + float32 + del df | Issue 2: RAM 32GB 환경 메모리 절감 |
| Hard cutoff -> MAD 순서 고정 | Issue 3: MAD trap 방지 |
| `n_genes >= 200`, `total_counts >= 500` 기준 | Issue 3, 4: 노이즈 세포 제거 |
| `harmonypy` 직접 호출 | Issue 6: `sce` 래퍼 shape 버그 우회 |
| `adata.raw = adata`를 HVG 직전에 | Issue 7: marker gene 전체 유전자 보존 |
| `n_pcs=17` | PCA elbow plot 기준 |
| `random_state=42` | UMAP 재현성 고정 |

최종 파이프라인 -> `01_geo_pipeline.ipynb`
